In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from myutils import *


In [19]:
train_data = pd.read_csv('Dataset/processed/train_processed/recoded_train.csv')
test_data = pd.read_csv('Dataset/processed/test_processed/recoded_test.csv')
train_data.head()
feature_json = read_jsonl('config/feature.json')
feature_config = feature_json['features']

In [20]:
test_data.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glyburide,pioglitazone,rosiglitazone,insulin,change,diabetesMed,readmitted,diag_1,diag_2,diag_3
0,2.0,0,0,NaN,NaN,0.0,1,NaN,37.0,41,...,1,1,1,1,1,0,0,4,NaN,NaN
1,0.0,0,4,0.0,0.0,12.0,9,NaN,NaN,47,...,1,1,1,2,1,1,2,4,1.0,5.0
2,NaN,0,5,1.0,0.0,9.0,2,NaN,NaN,66,...,1,1,1,0,0,1,0,1,1.0,1.0
3,0.0,0,6,0.0,0.0,12.0,6,NaN,NaN,87,...,3,1,1,2,0,1,0,7,2.0,4.0
4,2.0,0,7,0.0,8.0,12.0,1,NaN,3.0,28,...,1,1,1,0,0,1,2,1,1.0,4.0


In [21]:
features = train_data.columns.tolist()
print(f'{len(features)} features: {features}')

50 features: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'diag_1', 'diag_2', 'diag_3']


In [49]:
def show_var_relation(feature:str,data:pd.DataFrame,feature_config:dict,target:str):
    data = data.fillna(-1)
    vc = data[feature].value_counts()
    print(vc)
    mapping = feature_config[feature]['label_encoding']['encoding_mapping']
    id2name = {v: k for k, v in mapping.items()}
    print(id2name)
    statics = {'class':[],'mean':[],'std':[],'min':[],'max':[],'median':[],'mode':[],'count':[]}
    print("==========statics==========")
    for name,group in data.groupby(feature):
        statics['class'].append(name)
        statics['mean'].append(group[target].mean())
        statics['std'].append(group[target].std())
        statics['min'].append(group[target].min())
        statics['max'].append(group[target].max())
        statics['median'].append(group[target].median())
        statics['mode'].append(group[target].mode().iloc[0] if len(group[target].mode()) > 0 else None)
        statics['count'].append(group[target].count())
    df = pd.DataFrame(statics)
    print(df)
    # 计算相关系数
    corr = data[[feature, target]].corr()
    print(f"{feature} 与 {target} 的相关系数矩阵：")
    print(corr)
    print(f"{feature} 与 {target} 的皮尔逊相关系数为: {corr.loc[feature, target]}")
show_var_relation('race',train_data,feature_config,'readmitted')


race
 2.0    67348
 0.0    16956
-1.0     2027
 3.0     1835
 4.0     1353
 1.0      586
Name: count, dtype: int64
{0: 'AfricanAmerican', 1: 'Asian', 2: 'Caucasian', 3: 'Hispanic', 4: 'Other'}
==========statics==========
   class      mean       std  min  max  median  mode  count
0   -1.0  1.590035  0.639409    0    2     2.0     2   2027
1    0.0  1.420028  0.687271    0    2     2.0     2  16956
2    1.0  1.546075  0.663139    0    2     2.0     2    586
3    2.0  1.407377  0.686692    0    2     2.0     2  67348
4    3.0  1.473025  0.681351    0    2     2.0     2   1835
5    4.0  1.505543  0.670715    0    2     2.0     2   1353
race 与 readmitted 的相关系数矩阵：
                race  readmitted
race        1.000000   -0.014938
readmitted -0.014938    1.000000
race 与 readmitted 的皮尔逊相关系数为: -0.014938392550048243


In [51]:

feature = features[3]
show_var_relation(feature,train_data,feature_config,'readmitted')

gender
 0.0    48462
 1.0    41640
-1.0        3
Name: count, dtype: int64
{0: 'Female', 1: 'Male'}
==========statics==========
   class      mean       std  min  max  median  mode  count
0   -1.0  2.000000  0.000000    2    2     2.0     2      3
1    0.0  1.408815  0.686496    0    2     2.0     2  48462
2    1.0  1.427738  0.685324    0    2     2.0     2  41640
gender 与 readmitted 的相关系数矩阵：
              gender  readmitted
gender      1.000000    0.013667
readmitted  0.013667    1.000000
gender 与 readmitted 的皮尔逊相关系数为: 0.013667127725071388


In [53]:
feature = features[4]
show_var_relation(feature,train_data,feature_config,'readmitted')

age
7    22994
6    19943
5    15393
8    15027
4     8714
3     3389
9     2413
2     1464
1      624
0      144
Name: count, dtype: int64
{0: '[0-10)', 1: '[10-20)', 2: '[20-30)', 3: '[30-40)', 4: '[40-50)', 5: '[50-60)', 6: '[60-70)', 7: '[70-80)', 8: '[80-90)', 9: '[90-100)'}
==========statics==========
   class      mean       std  min  max  median  mode  count
0      0  1.819444  0.420668    0    2     2.0     2    144
1      1  1.562500  0.607025    0    2     2.0     2    624
2      2  1.409836  0.723636    0    2     2.0     2   1464
3      3  1.457952  0.688765    0    2     2.0     2   3389
4      4  1.446179  0.678510    0    2     2.0     2   8714
5      5  1.457611  0.666360    0    2     2.0     2  15393
6      6  1.418593  0.685011    0    2     2.0     2  19943
7      7  1.388232  0.691709    0    2     2.0     2  22994
8      8  1.377720  0.696345    0    2     2.0     2  15027
9      9  1.464981  0.694175    0    2     2.0     2   2413
age 与 readmitted 的相关系数矩阵：
     

In [56]:
feature = features[5]
show_var_relation(feature,train_data,feature_config,'readmitted')

weight
-1.0    87307
 8.0     1170
 7.0      781
 2.0      555
 3.0      128
 6.0       75
 1.0       44
 4.0       31
 5.0       11
 0.0        3
Name: count, dtype: int64
{0: '>200', 1: '[0-25)', 2: '[100-125)', 3: '[125-150)', 4: '[150-175)', 5: '[175-200)', 6: '[25-50)', 7: '[50-75)', 8: '[75-100)'}
==========statics==========
   class      mean       std  min  max  median  mode  count
0   -1.0  1.421627  0.686489    0    2     2.0     2  87307
1    0.0  1.000000  0.000000    1    1     1.0     1      3
2    1.0  1.045455  0.608259    0    2     1.0     1     44
3    2.0  1.302703  0.662489    0    2     1.0     1    555
4    3.0  1.359375  0.624114    0    2     1.0     1    128
5    4.0  1.354839  0.660726    0    2     1.0     1     31
6    5.0  1.363636  0.674200    0    2     1.0     1     11
7    6.0  1.386667  0.655400    0    2     1.0     2     75
8    7.0  1.325224  0.678793    0    2     1.0     2    781
9    8.0  1.257265  0.647138    0    2     1.0     1   1170
weight 